# Part 2: Hybrid triage demo with IBM Granite on LinuxONE

In [1]:
import platform, os, time, re, json, gc, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightgbm as lgb
torch.set_num_threads(os.cpu_count())

with open('model_meta.json') as f:
    meta = json.load(f)
feature_cols, lab_cols, vital_cols = meta['feature_cols'], meta['lab_cols'], meta['vital_cols']
emb_cols, tier1_cols = meta['emb_cols'], meta['tier1_cols']
mu, sd = pd.Series(meta['mu']), pd.Series(meta['sd'])
THR1, THR2 = meta['thr1'], meta['thr2']

lgb_model = lgb.Booster(model_file='tier1_lgb.txt')
lgb_vitals = lgb.Booster(model_file='tier2_lgb.txt')

class Autoencoder(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
                                     nn.Linear(64, embedding_dim), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Linear(64, input_dim))
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

ae = Autoencoder(len(feature_cols), meta['EMB_DIM'])
ae.load_state_dict(torch.load('ae.pt'))
ae.eval()

demo = pd.read_csv('demo_patients.csv')
print(f"Models loaded on {platform.machine()} | alert thresholds: Tier 1 {THR1:.3f}, Tier 2 {THR2:.3f}")
print(f"Demo patients: {len(demo)} unseen test rows")
!grep MemAvailable /proc/meminfo

Models loaded on s390x | alert thresholds: Tier 1 0.508, Tier 2 0.515
Demo patients: 6 unseen test rows
MemAvailable:    6498432 kB


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GRANITE_ID = "ibm-granite/granite-3.1-1b-a400m-instruct"   # small + fast; for the bigger one use "ibm-granite/granite-3.3-2b-instruct"
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(GRANITE_ID)
granite = AutoModelForCausalLM.from_pretrained(GRANITE_ID, dtype=torch.bfloat16, low_cpu_mem_usage=True)
granite.eval()
print(f"Granite loaded on {platform.machine()} in {time.perf_counter() - t0:.0f} s")
!grep MemAvailable /proc/meminfo

def call_granite(prompt, max_new_tokens=80):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = granite.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Granite loaded on s390x in 31 s
MemAvailable:    3726052 kB


In [12]:
def tier1_score(raw_rows):
    raw_rows = raw_rows[feature_cols].astype(float)
    xn = ((raw_rows - mu) / sd).fillna(0)
    with torch.no_grad():
        z, _ = ae(torch.tensor(xn[feature_cols].values, dtype=torch.float32))
    feats = pd.concat([raw_rows.reset_index(drop=True), pd.DataFrame(z.numpy(), columns=emb_cols)], axis=1)
    return lgb_model.predict(feats[tier1_cols])

def concerning_readings(r):
    # Standard bedside screening cut-offs (SIRS / qSOFA style), checked in CODE
    checks = [('HR',    lambda v: v > 90,             'heart rate {:.0f} bpm (above 90)'),
              ('Resp',  lambda v: v >= 22,            'breathing rate {:.0f}/min (22 or more)'),
              ('Temp',  lambda v: v > 38 or v < 36,   'temperature {:.1f} C (outside 36-38)'),
              ('SBP',   lambda v: v <= 100,           'systolic BP {:.0f} mmHg (100 or less)'),
              ('MAP',   lambda v: v < 65,             'mean arterial pressure {:.0f} mmHg (below 65)'),
              ('O2Sat', lambda v: v < 92,             'oxygen saturation {:.0f}% (below 92)')]
    return [text.format(r[c]) for c, bad, text in checks if c in r and pd.notna(r[c]) and bad(r[c])]

def nurse_note(alert, readings):
    flags = concerning_readings(readings)
    summary = ("Abnormal readings: " + "; ".join(flags) + ".") if flags \
              else "All recorded vital signs are within standard screening ranges."
    if alert:
        action = "NEXT STEP: Sepsis alert - request urgent review by a doctor."
    elif flags:
        action = "NEXT STEP: No alert - continue routine monitoring and re-check vitals within the hour."
    else:
        action = "NEXT STEP: No alert - continue routine monitoring."
    return f"{summary}\n{action}"

def triage(raw_row, explain=False):
    raw_row = raw_row[feature_cols].astype(float)
    if raw_row[lab_cols].notna().any(axis=1).iloc[0]:
        tier, score, thr = "Tier 1 (hospital: vitals + labs)", tier1_score(raw_row)[0], THR1
    else:
        tier, score, thr = "Tier 2 (clinic: bedside vitals only)", lgb_vitals.predict(raw_row[vital_cols])[0], THR2
    alert = score >= thr
    print(f"ROUTE: {tier} -> LinuxONE inference")
    print(f"Risk score {score*100:.1f}% vs alert threshold {thr*100:.1f}% -> {'ALERT' if alert else 'no alert'}")
    if explain:
        readings = {c: float(v) for c, v in raw_row[vital_cols].iloc[0].items() if pd.notna(v)}
        print("NURSE NOTE:", nurse_note(alert, readings))
    return score, alert

In [15]:
for i in range(len(demo)):
    r = demo.iloc[[i]]
    print(f"\n--- Test patient {int(r['Patient_ID'].iloc[0])}, hour {int(r['Hour'].iloc[0])} "
          f"(actual: {'SEPSIS' if r['SepsisLabel'].iloc[0] == 1 else 'no sepsis'}) ---")
    triage(r)

print("\n--- First patient again, but at a clinic with NO lab tests (labs removed) -> Tier 2 + nurse note ---")
rural = demo.iloc[[0]].copy()
rural[lab_cols] = np.nan
_ = triage(rural, explain=True)


--- Test patient 8250, hour 22 (actual: SEPSIS) ---
ROUTE: Tier 1 (hospital: vitals + labs) -> LinuxONE inference
Risk score 11.2% vs alert threshold 50.8% -> no alert

--- Test patient 117517, hour 42 (actual: SEPSIS) ---
ROUTE: Tier 1 (hospital: vitals + labs) -> LinuxONE inference
Risk score 61.6% vs alert threshold 50.8% -> ALERT

--- Test patient 111891, hour 6 (actual: SEPSIS) ---
ROUTE: Tier 1 (hospital: vitals + labs) -> LinuxONE inference
Risk score 17.2% vs alert threshold 50.8% -> no alert

--- Test patient 5912, hour 2 (actual: no sepsis) ---
ROUTE: Tier 2 (clinic: bedside vitals only) -> LinuxONE inference
Risk score 42.4% vs alert threshold 51.5% -> no alert

--- Test patient 6979, hour 7 (actual: no sepsis) ---
ROUTE: Tier 1 (hospital: vitals + labs) -> LinuxONE inference
Risk score 3.4% vs alert threshold 50.8% -> no alert

--- Test patient 107126, hour 6 (actual: no sepsis) ---
ROUTE: Tier 1 (hospital: vitals + labs) -> LinuxONE inference
Risk score 9.0% vs alert thre

In [14]:
def triage_from_text(note):
    t0 = time.perf_counter()
    raw = call_granite(
        "Extract the patient's data from this note as JSON with keys "
        "Age, Gender (1 = male, 0 = female), HR, Resp, Temp (Celsius), SBP, DBP, O2Sat. "
        "Use null for anything not mentioned. Do not calculate or guess values. Output only the JSON.\n"
        f"Note: {note}", max_new_tokens=90)
    print("GRANITE EXTRACTED:", raw, f"\n(took {time.perf_counter() - t0:.0f} s)")
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        print("Could not read Granite's output as JSON.")
        return
    data = json.loads(m.group())
    g = data.get('Gender')
    if isinstance(g, str):
        data['Gender'] = 1 if g.strip().lower().startswith('m') else 0
    sbp, dbp = pd.to_numeric(data.get('SBP'), errors='coerce'), pd.to_numeric(data.get('DBP'), errors='coerce')
    data['MAP'] = (sbp + 2 * dbp) / 3 if pd.notna(sbp) and pd.notna(dbp) else None   # calculated in code, not by Granite
    print(f"MAP calculated in code: {data['MAP']:.0f} mmHg" if data['MAP'] is not None else "MAP: not available")
    row = pd.DataFrame([{c: pd.to_numeric(data.get(c), errors='coerce') for c in feature_cols}])
    triage(row, explain=True)   # no labs in a typed note -> Tier 2

triage_from_text("38 year old male, heart rate 118, breathing 26 per minute, temp 38.9, BP 92/55, sats 92%")

GRANITE EXTRACTED: {
  "Age": 38,
  "Gender": 1,
  "HR": 118,
  "Resp": 26,
  "Temp": 38.9,
  "SBP": 92,
  "DBP": 55,
  "O2Sat": 92
} 
(took 152 s)
MAP calculated in code: 67 mmHg
ROUTE: Tier 2 (clinic: bedside vitals only) -> LinuxONE inference
Risk score 86.0% vs alert threshold 51.5% -> ALERT
NURSE NOTE: Abnormal readings: heart rate 118 bpm (above 90); breathing rate 26/min (22 or more); temperature 38.9 C (outside 36-38); systolic BP 92 mmHg (100 or less).
NEXT STEP: Sepsis alert - request urgent review by a doctor.
